# Урок 11. Regularization — лекарство от переобучения
**Библиотеки:** numpy, sklearn, matplotlib

**Цель:** понять Ridge/Lasso и параметр alpha.

## Простыми словами
Переобученная модель обычно имеет ОГРОМНЫЕ веса — она выкручивает их, чтобы попасть в каждую
шумную точку. Регуляризация добавляет в cost штраф за большие веса:

> новый cost = старая ошибка + alpha * (размер весов)

Модель теперь балансирует: «попадать в точки» против «держать веса скромными».
- **Ridge (L2):** штраф = сумма квадратов весов. Прижимает все веса плавно.
- **Lasso (L1):** штраф = сумма модулей. Может занулить вес полностью → автоотбор признаков.
- **alpha** — сила штрафа: 0 = обычная регрессия, огромный = модель почти константа (underfit).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

rng = np.random.default_rng(2)
X = np.sort(rng.uniform(0, 4, 40)).reshape(-1, 1)
y = 1.5 * X[:, 0] ** 2 - 2 * X[:, 0] + rng.normal(0, 2.0, 40)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=1)

xs = np.linspace(0, 4, 200).reshape(-1, 1)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
models = [("Polynomial deg=12 БЕЗ регуляризации", LinearRegression()),
          ("Ridge alpha=1", Ridge(alpha=1)),
          ("Ridge alpha=100", Ridge(alpha=100))]
for ax, (name, reg) in zip(axes, models):
    # pipeline: полиномы 12 степени (нарочно переусложняем) -> scaler -> модель
    pipe = make_pipeline(PolynomialFeatures(12), StandardScaler(), reg).fit(X_tr, y_tr)
    ax.scatter(X_tr, y_tr, s=15)
    ax.plot(xs, pipe.predict(xs), 'r')
    ax.set_title(f'{name}\ntest R2 = {r2_score(y_te, pipe.predict(X_te)):.2f}')
    ax.set_ylim(-8, 22)
plt.show()

In [ ]:
# Как alpha влияет на веса Ridge
for a in [0, 1, 10, 1000]:
    pipe = make_pipeline(PolynomialFeatures(12), StandardScaler(), Ridge(alpha=max(a, 1e-10)))
    pipe.fit(X_tr, y_tr)
    w = pipe.named_steps['ridge'].coef_
    print(f"alpha={a:>5}: max|w| = {np.abs(w).max():8.1f}, test R2 = {r2_score(y_te, pipe.predict(X_te)):.2f}")

## Что произошло
Полином 12-й степени без штрафа дико извивается между точками (overfit, веса гигантские).
Ridge alpha=1 — кривая успокоилась, test R2 вырос. Alpha=100 — модель стала слишком ленивой
(начинается underfit). Видно по таблице: чем больше alpha, тем меньше веса.

Alpha — «ручка громкости» сложности. Подбирают перебором по validation (GridSearchCV).

## Типичные ошибки
1. Применять Ridge/Lasso без масштабирования — штраф несправедливо бьёт по признакам с большими числами.
2. Думать, что регуляризация всегда улучшает — при underfitting она вредит.
3. Путать alpha регуляризации с learning rate (тоже alpha) — это РАЗНЫЕ вещи.

## Конспект 📓
Регуляризация = штраф за большие веса в cost. Ridge(L2) — прижимает все, Lasso(L1) — зануляет.
alpha больше → модель проще. Обязательно scaling. Лечит overfit, при underfit вредна.

---
## Мини-задание 💪
Замени Ridge на Lasso(alpha=0.1) и выведи coef_. Сколько весов стало ровно 0? Это и есть отбор признаков.

## 5 проверочных вопросов ❓
1. Почему у переобученной модели обычно огромные веса?
2. Что добавляет регуляризация в cost function?
3. Ridge vs Lasso — главное отличие?
4. Что будет при alpha=0 и при alpha=10^6?
5. Почему перед регуляризацией нужен StandardScaler?

## Домашнее задание 🏠
Задача 26 из practice_tasks.md. Через GridSearchCV найди лучший alpha для Ridge на данных урока (сетка: 0.01, 0.1, 1, 10, 100).
